# LoRA × GPT-2 Medium — E2E NLG / WikiText

This notebook fine-tunes `gpt2-medium` with the `project_LoRA` repo's from-scratch LoRA implementation, targeting the GPT-2 row of **Table 3** from [Hu et al., 2021](https://arxiv.org/abs/2106.09685).

**Setup (paper Table 3 / 11, GPT-2 Medium + LoRA on E2E NLG):**
- Target modules: `W_q`, `W_v` slices of the fused `c_attn` Conv1D (handled by `LoRAConv1DQV`)
- Rank `r = 4`, scaling `α = 32`, dropout 0.1
- Optimizer: AdamW, weight decay 0.01, linear schedule, 6% warmup
- Trainable params: ~0.4M (LoRA adapters only — lm_head stays tied/frozen)

**Runtime budget (Colab T4, free tier):**
- WikiText-2 (3 epochs, block 512): ~10–20 min — fast sanity check.
- E2E NLG (5 epochs, batch 8, max_len 512): ~1–2 h.

In [ ]:
!nvidia-smi

## 1. Load the project code

Pick **one** of the two options below.

**Option A (recommended):** zip the `code/` folder from your repo and upload it here.
```bash
# on your laptop, from inside project_LoRA/:
zip -r code.zip code -x 'code/external/*' -x '*/__pycache__/*' -x '*/.venv/*'
```

**Option B:** clone the full repo from a public GitHub mirror.

In [ ]:
# --- Option A: upload code.zip ---
from google.colab import files
import os, zipfile

uploaded = files.upload()  # drag-drop code.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

def find_code_root(base="/content", max_depth=2):
    for dirpath, _, filenames in os.walk(base):
        depth = dirpath[len(base):].count(os.sep)
        if depth > max_depth:
            continue
        if "requirements.txt" in filenames and os.path.isdir(os.path.join(dirpath, "lora")):
            return dirpath
    return None

root = find_code_root()
assert root is not None, "Couldn't find a folder with requirements.txt + lora/ in the uploaded zip."
os.chdir(root)
print("Working dir:", os.getcwd())
print(os.listdir(root))

In [ ]:
# --- Option B: clone from a public GitHub mirror (uncomment if applicable) ---
# !git clone https://github.com/austinlu2005/myLoRA.git /content/myLoRA
# %cd /content/myLoRA/code

In [ ]:
!pip install -q -r requirements.txt

## 2. Imports

In [ ]:
import sys, os, json, math, time
sys.path.insert(0, os.getcwd())

import torch
from transformers import AutoTokenizer

from models.gpt2_wrapper import build_gpt2_lora
from dataloaders.e2e_nlg import load_e2e_nlg
from dataloaders.wikitext import load_wikitext
from training.trainer import Trainer
from training.optim import build_optimizer, build_scheduler
from lora.merge import merge_lora, unmerge_lora
from utils.seed import set_seed
from utils.param_utils import print_trainable_parameters

print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 3. Hyperparameters

Paper-aligned settings for GPT-2 Medium + LoRA on E2E NLG (Table 11). WikiText is a sanity-check setup, not in the paper.

In [ ]:
COMMON = {
    "rank": 4, "alpha": 32, "dropout": 0.1,
    "weight_decay": 0.01, "warmup_ratio": 0.06,
    "seed": 42, "grad_clip": 1.0,
}

TASK_HPARAMS = {
    "e2e_nlg":  {"lr": 2e-4, "batch_size": 8, "num_epochs": 5, "max_length": 512},
    "wikitext": {"lr": 2e-4, "batch_size": 4, "num_epochs": 3, "max_length": 512,
                  "config": "wikitext-2-raw-v1"},
}

# LoRA paper Table 3, GPT-2 Medium + LoRA on E2E NLG (test split)
PAPER_E2E = {"BLEU": 70.4, "NIST": 8.85, "METEOR": 46.8, "ROUGE-L": 71.8, "CIDEr": 2.53}

## 4. Training function

Builds GPT-2 with LoRA on `c_attn` Q/V slices, loads the appropriate dataset, and trains with `task_type="causal_lm"` (so eval reports perplexity instead of running argmax).

In [ ]:
RUNS_DIR = "/content/runs_gpt2"
os.makedirs(RUNS_DIR, exist_ok=True)

def train_gpt2(task, hparams, common=COMMON, runs_dir=RUNS_DIR, model_name="gpt2-medium"):
    set_seed(common["seed"])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model, replaced = build_gpt2_lora(
        model_name=model_name,
        rank=common["rank"], alpha=common["alpha"], dropout=common["dropout"],
    )
    print(f"[{task}] injected LoRA into {len(replaced)} c_attn modules")
    print_trainable_parameters(model)

    if task == "e2e_nlg":
        data = load_e2e_nlg(tokenizer, max_length=hparams["max_length"])
    elif task == "wikitext":
        data = load_wikitext(tokenizer, block_size=hparams["max_length"], config=hparams["config"])
    else:
        raise ValueError(task)
    train_ds, eval_ds = data["train"], data["validation"]
    print(f"[{task}] train={len(train_ds)}  eval={len(eval_ds)}")

    steps_per_epoch = math.ceil(len(train_ds) / hparams["batch_size"])
    num_training_steps = steps_per_epoch * hparams["num_epochs"]

    optimizer = build_optimizer(model, lr=hparams["lr"], weight_decay=common["weight_decay"])
    scheduler = build_scheduler(optimizer, num_training_steps, warmup_ratio=common["warmup_ratio"])

    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = f"{runs_dir}/{task}"
    trainer = Trainer(
        model=model, train_dataset=train_ds, eval_dataset=eval_ds,
        optimizer=optimizer, scheduler=scheduler,
        batch_size=hparams["batch_size"], num_epochs=hparams["num_epochs"],
        device=device, compute_metrics=None,
        log_steps=100, grad_clip=common["grad_clip"], output_dir=out_dir,
        task_type="causal_lm",
    )

    t0 = time.time()
    trainer.train()
    wall = time.time() - t0

    best_ppl = min((h["perplexity"] for h in trainer.history if "perplexity" in h), default=None)
    result = {
        "task": task, "best_perplexity": best_ppl,
        "wall_seconds": wall, "history": trainer.history,
        "hparams": hparams, "common": common,
    }
    with open(f"{out_dir}/result.json", "w") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"[{task}] best perplexity = {best_ppl:.3f}  | wall {wall/60:.1f} min")
    return trainer, tokenizer, result

## 5. Sanity run: WikiText-2

Tiny dataset, runs in ~10–20 min on T4. Use this to confirm the full pipeline (LoRA injection → training → perplexity eval) before launching the longer E2E run.

In [ ]:
trainer_wt, tok_wt, res_wt = train_gpt2("wikitext", TASK_HPARAMS["wikitext"])

## 6. Main run: E2E NLG

5 epochs at batch 8, max_len 512. ~1–2 hours on T4. If you OOM, drop `batch_size` to 4 — LoRA-only training keeps gradients tiny, but activations at fp32 are the budget.

In [ ]:
trainer_e2e, tok_e2e, res_e2e = train_gpt2("e2e_nlg", TASK_HPARAMS["e2e_nlg"])

## 7. Sample generations from the fine-tuned model

Merge the LoRA adapters into the base weights for zero-overhead inference, then greedy-decode a few E2E meaning representations. The model should produce fluent restaurant descriptions if training worked.

In [ ]:
from dataloaders.e2e_nlg import PROMPT_SEP

model = trainer_e2e.model
tokenizer = tok_e2e
merge_lora(model)  # fold LoRA ΔW into base weights
model.eval()

examples = [
    "name[The Vaults], eatType[pub], priceRange[more than £30], customer rating[5 out of 5], near[Café Adriatic]",
    "name[Blue Spice], eatType[coffee shop], food[Indian], area[city centre], familyFriendly[no]",
    "name[The Eagle], eatType[restaurant], food[French], priceRange[cheap], area[riverside]",
]

for mr in examples:
    prompt = f"{mr}{PROMPT_SEP}"
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids,
            max_new_tokens=80,
            num_beams=5,
            no_repeat_ngram_size=3,
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    print("MR :", mr)
    print("GEN:", completion.strip())
    print("-" * 70)

unmerge_lora(model)  # restore so further training/saving still sees pure base + adapters

## 8. Training curves

In [ ]:
import matplotlib.pyplot as plt

completed = [t for t in ("wikitext", "e2e_nlg") if os.path.isfile(f"{RUNS_DIR}/{t}/result.json")]
if completed:
    fig, axes = plt.subplots(1, len(completed), figsize=(5*len(completed), 3.2), squeeze=False)
    for ax, t in zip(axes[0], completed):
        with open(f"{RUNS_DIR}/{t}/result.json") as f:
            hist = json.load(f)["history"]
        xs = [h["epoch"] for h in hist]
        ax.plot(xs, [h["eval_loss"] for h in hist], marker="o", label="eval loss")
        ax.plot(xs, [h["perplexity"] for h in hist], marker="s", label="perplexity")
        ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No completed runs yet.")

## 9. Notes & caveats

- **What's reported.** This notebook reports perplexity, which is what the trainer's causal-LM eval computes. The paper's Table 3 reports BLEU / NIST / METEOR / ROUGE-L / CIDEr on E2E NLG — those need a separate generation-and-scoring pass (decode the eval set with beam=10 and run e.g. `nltk` BLEU + `pycocoevalcap` for the rest). `evaluation/generation_metrics.py` is the place to hook that up.
- **`LoRAConv1DQV` vs full `c_attn`.** GPT-2's `c_attn` is the fused `[Q | K | V]` Conv1D. The paper applies LoRA only to W_q and W_v. `inject_lora(..., conv1d_qv=True)` (which `build_gpt2_lora` passes) swaps `c_attn` for `LoRAConv1DQV`, which holds independent rank-`r` adapters on Q and V and leaves K untouched.
- **Single seed.** Paper reports median of 3 seeds for GPT-2. Loop `common["seed"]` over `{0, 1, 2}` and take the median if you want to match.
- **Saving.** `Trainer` saves only the LoRA state (`lora_best.pt`, ~1.5 MB) when eval loss improves — not the 354M base model.
- **Persisting through Colab disconnects.**

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/runs_gpt2 /content/drive/MyDrive/lora_runs_gpt2
```

## 10. Resume training from a saved adapter

Continue training from a previously saved `lora_best.pt` for another N epochs. Useful when the eval-loss curve was still trending down at the end of the original run, or when you've added training-side fixes (e.g., label smoothing) and want to keep the adapter's progress.

**Caveats:**
- The trainer saves *adapter weights only*, not optimizer state. Resuming creates a fresh AdamW optimizer — Adam moments are lost. Expect a brief loss spike for ~50–100 steps before the optimizer re-settles. This is a *warm restart*, not a true checkpoint resume.
- Use a **smaller LR** (~half the original) for resumption. Full LR risks undoing learned weights during the warm restart.
- **Skip warmup** (or use a tiny one). Warmup is for cold initialization; here we want the LR up immediately.
- The adapter from this resume run is saved separately at `{RUNS_DIR}/e2e_nlg_resumed/lora_best.pt` — your original `runs_gpt2/e2e_nlg/lora_best.pt` is untouched.


In [ ]:
# Upload the previous lora_best.pt
from google.colab import files
import shutil

uploaded = files.upload()  # drag-drop lora_best.pt
lora_name = next(iter(uploaded))
RESUME_PATH = "/content/lora_best.pt"
shutil.move(lora_name, RESUME_PATH)
print(f"Adapter at: {RESUME_PATH}")


In [ ]:
from lora.save_load import load_lora_state_dict

# --- resume hyperparameters ---
RESUME_EPOCHS       = 5
RESUME_LR           = 1e-4   # half of original 2e-4 (adapter is partly trained)
RESUME_WARMUP_RATIO = 0.0    # no fresh warmup needed
RESUME_RUN_DIR      = f"{RUNS_DIR}/e2e_nlg_resumed"
os.makedirs(RESUME_RUN_DIR, exist_ok=True)

set_seed(COMMON["seed"])

# 1. Tokenizer + fresh GPT-2 + LoRA injection
tokenizer = AutoTokenizer.from_pretrained("gpt2-medium")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model, replaced = build_gpt2_lora(
    model_name="gpt2-medium",
    rank=COMMON["rank"], alpha=COMMON["alpha"], dropout=COMMON["dropout"],
)
print(f"Injected LoRA into {len(replaced)} c_attn modules")

# 2. Overwrite freshly-initialized adapter weights with the saved checkpoint
load_lora_state_dict(model, RESUME_PATH)
print(f"Loaded adapter from {RESUME_PATH}")
print_trainable_parameters(model)

# 3. Reload data
hp = TASK_HPARAMS["e2e_nlg"]
data = load_e2e_nlg(tokenizer, max_length=hp["max_length"])
train_ds, eval_ds = data["train"], data["validation"]
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

# 4. Fresh optimizer + scheduler (Adam state lost — warm restart)
steps_per_epoch = math.ceil(len(train_ds) / hp["batch_size"])
num_training_steps = steps_per_epoch * RESUME_EPOCHS
optimizer = build_optimizer(model, lr=RESUME_LR, weight_decay=COMMON["weight_decay"])
scheduler = build_scheduler(optimizer, num_training_steps, warmup_ratio=RESUME_WARMUP_RATIO)

device = "cuda" if torch.cuda.is_available() else "cpu"
trainer_resume = Trainer(
    model=model, train_dataset=train_ds, eval_dataset=eval_ds,
    optimizer=optimizer, scheduler=scheduler,
    batch_size=hp["batch_size"], num_epochs=RESUME_EPOCHS,
    device=device, compute_metrics=None,
    log_steps=100, grad_clip=COMMON["grad_clip"], output_dir=RESUME_RUN_DIR,
    task_type="causal_lm",
)

t0 = time.time()
trainer_resume.train()
wall = time.time() - t0

best_ppl = min((h["perplexity"] for h in trainer_resume.history if "perplexity" in h), default=None)
print(f"\nResume done | wall {wall/60:.1f} min | best ppl: {best_ppl:.3f}")
print(f"New adapter saved when eval improved: {RESUME_RUN_DIR}/lora_best.pt")

with open(f"{RESUME_RUN_DIR}/result.json", "w") as f:
    json.dump({
        "history": trainer_resume.history,
        "resume_lr": RESUME_LR,
        "resume_epochs": RESUME_EPOCHS,
        "loaded_from": RESUME_PATH,
        "best_perplexity": best_ppl,
    }, f, indent=2, default=str)
